In [ ]:
pip install pandas numpy nltk scikit-learn plotly seaborn transformers torch wordcloud

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import plotly.express as px
import plotly.graph_objects as go

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import pipeline

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
df = pd.read_csv("mtsamples.csv")

In [ ]:
df = df[['transcription', 'medical_specialty']]

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.rename(columns={
    'transcription': 'text',
    'medical_specialty': 'label'
}, inplace=True)

In [ ]:
df.head()

,text,label
0,"SUBJECTIVE:, This 23-year-old white female pr...",Allergy / Immunology
1,"PAST MEDICAL HISTORY:, He has difficulty climb...",Bariatrics
2,"HISTORY OF PRESENT ILLNESS: , I have seen ABC ...",Bariatrics
3,"2-D M-MODE: , ,1. Left atrial enlargement wit...",Cardiovascular / Pulmonary
4,1. The left ventricular cavity size and wall ...,Cardiovascular / Pulmonary


In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['clean_text'] = df['text'].apply(clean_text)

In [ ]:
fig = px.histogram(
    df,
    x='label',
    color='label',
    title="Distribution of Medical Specialties",
    color_discrete_sequence=px.colors.qualitative.Bold
)

fig.update_layout(xaxis_title="Specialty", yaxis_title="Count")
fig.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'],
    df['label'],
    test_size=0.2,
    random_state=42
)

In [ ]:
# Step 1: Sample data
df = df.sample(2000, random_state=42)

# Step 2: Shorten text
df['short_text'] = df['text'].str[:200]

# Step 3: Fast model
classifier = pipeline(
    "zero-shot-classification",
    model="typeform/distilbert-base-uncased-mnli",
    device=-1
)

# Step 4: Batch prediction
df['clinical_tag'] = batch_predict(df['short_text'].tolist(), batch_size=64)

config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

100%|██████████| 32/32 [18:16<00:00, 34.28s/it]


In [ ]:
fig = px.pie(
    df,
    names='clinical_tag',
    title="Clinical Insight Categories",
    color_discrete_sequence=px.colors.sequential.RdBu
)

fig.show()

In [ ]:
from collections import Counter

all_words = " ".join(df['clean_text']).split()
word_freq = Counter(all_words).most_common(20)

words_df = pd.DataFrame(word_freq, columns=['word', 'count'])

fig = px.bar(
    words_df,
    x='word',
    y='count',
    title="Top 20 Frequent Medical Terms",
    color='count',
    color_continuous_scale='Viridis'
)

fig.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression()
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)

print(classification_report(y_test, y_pred))

                                precision    recall  f1-score   support

          Allergy / Immunology       0.00      0.00      0.00         2
                       Autopsy       0.00      0.00      0.00         1
                    Bariatrics       0.00      0.00      0.00         2
    Cardiovascular / Pulmonary       0.28      0.28      0.28        79
                  Chiropractic       0.00      0.00      0.00         2
    Consult - History and Phy.       0.28      0.53      0.37       111
    Cosmetic / Plastic Surgery       0.00      0.00      0.00         7
                     Dentistry       0.00      0.00      0.00         8
                   Dermatology       0.00      0.00      0.00         2
          Diets and Nutritions       0.00      0.00      0.00         1
             Discharge Summary       0.33      0.26      0.29        23
          ENT - Otolaryngology       0.00      0.00      0.00        32
        Emergency Room Reports       0.00      0.00      0.00  

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



In [ ]:
cm = confusion_matrix(y_test, y_pred)
labels = list(set(y_test))

fig = go.Figure(data=go.Heatmap(
    z=cm,
    x=labels,
    y=labels,
    colorscale='Blues'
))

fig.update_layout(
    title="Confusion Matrix",
    xaxis_title="Predicted",
    yaxis_title="Actual"
)

fig.show()

In [ ]:
fig = px.histogram(
    df,
    x='clinical_tag',
    color='clinical_tag',
    title="Clinical Insight Overview",
    color_discrete_sequence=px.colors.qualitative.Dark24
)

fig.show()